[Reference](https://levelup.gitconnected.com6-claude-skills-mcp-tools-that-helped-me-to-maintain-5000-files-monorepo-c5a90bb657bc?sk=11fd1345f6bda36fcff366eb4f6eb7e0$0)

# 1. Debugging CI/CD: Jenkins + Github Actions Integrations


## Skill Prompt
```
## Skill: CI/CD Failure Validator
**Input**
- pr_url / pr_number
**Flow**
1. Fetch PR checks (GitHub / Jenkins)
2. If all pass → exit
3. Else → analyze failures:
- GitHub Actions:
  - Get failed step + logs
  - Extract first meaningful error
- Jenkins:
  - Fetch build status + logs via MCP
  - Identify failing stage + root error
**Output**
- Failed checks
- Concise RCA per failure
**Rules**
- No hallucination
- Focus on first real error
- Keep concise
```

## Jenkins MCP Tool:

In [1]:
from fastapi import FastAPI
import requests, os

app = FastAPI()
BASE = os.getenv("JENKINS_URL")
AUTH = (os.getenv("JENKINS_USER"), os.getenv("JENKINS_TOKEN"))
def _get(url):
    return requests.get(url, auth=AUTH)
@app.get("/build/status")
def build_status(job: str, build: int | None = None):
    url = f"{BASE}/job/{job}" + (f"/{build}" if build else "") + "/api/json"
    data = _get(url).json()
    return {
        "job": job,
        "status": data.get("result"),
        "building": data.get("building"),
        "url": data.get("url"),
    }
@app.get("/build/logs")
def build_logs(job: str, build: int):
    url = f"{BASE}/job/{job}/{build}/consoleText"
    logs = _get(url).text[-5000:]
    return {
        "job": job,
        "build": build,
        "logs": logs
    }

# 2. Designing Safe & Scalable Django Migrations
```
## Skill: Safe Django Migrations
**Goal**
Design and execute Django migrations safely for large-scale databases
**Strategy**
1. Add field (nullable)
2. Backfill in batches
3. Apply constraints (NOT NULL, index)
## Capabilities
- Risk Detection: Identify lock-prone and heavy operations  
- Step-wise Strategy: Break migrations into safe, incremental steps  
- Batch Processing: Backfill data using chunked queries  
- Lock Avoidance: Suggest safer alternatives (e.g., concurrent indexes)  
- Execution Planning: Recommend manual runs for critical migrations
```

# 3. Create a PR with Description, Files Changes, and Impact
```
# PR Creation Skill
## Description
Generate a well-structured Pull Request description by analyzing code changes, summarizing intent, and filling in all relevant sections to improve review efficiency and clarity.
---
## Summary of Changes
Provide a concise overview of what this PR does:
- What problem is being solved?
- What is the approach taken?
- Any important context reviewers should know?
---
## How to Test
Step-by-step instructions for verifying the changes:
1. Setup steps (if any)
2. Run commands
3. Expected outcomes
```

# 4. Write Test Cases
```
# Test Case Writing Skill
## Description
Generate high-quality, maintainable, and comprehensive test cases by analyzing the codebase. Ensure strict adherence to best practices such as no patching, proper inheritance from base test utilities, and achieving near-complete test coverage.
---
## Core Principles
- **Do Not Patch Code**
  - Never modify or patch the original implementation.
  - Tests must validate behavior without altering source logic.
- **Use Base Test Class**
  - Always inherit from the shared base test class located in `utils`.
  - Example:
    ```python
    from utils.base_test import BaseTestClass
    class TestFeature(BaseTestClass):
        ...
    ```
---
## Test Coverage Requirements
- Ensure **minimum 99% test coverage**
---
## Types of Test Cases
### 1. Unit Tests
- Test individual functions/methods in isolation
- Mock external dependencies
- Validate:
  - Inputs → Outputs
  - Edge behaviors
  - Error handling
### 2. Integration Tests
- Validate interaction between components
- Use real or semi-real dependencies (DB, APIs, services)
- Ensure end-to-end correctness of workflows
---
## Edge Case Coverage
Explicitly include:
- Null / None inputs
- Empty inputs (lists, strings, dicts)
- Boundary values (min/max limits)
- Invalid data types
- Large inputs / stress scenarios
- Failure scenarios (timeouts, exceptions, retries)
---
## Test Structure Guidelines
- Follow clear Arrange → Act → Assert pattern
- Keep tests deterministic and independent
- Avoid flaky tests (no random/uncontrolled dependencies)
## Naming Conventions
Use descriptive test names:
`test_<function>_<scenario>_<expected_result>`
Example:
`test_create_user_invalid_email_raises_error`
---
## Assertions and Validation
Use strict and meaningful assertions
Validate:
 - Return values
 - Side effects
 - State changes
 - Raised exceptions
 - Test Data Management
 - Use reusable fixtures or factory methods
 ```

# 5. JIRA Ticket with Description, QA Test Cases, Acceptance Criteria
```
## Skill: JIRA Ticket Generator
**Fields**
- Project (auto-detect)
- Title, Type, Priority
**Include**
- Description + Impact
- RCA (root cause + gap)
- Fix (approach)
- QA Test Cases
- Acceptance Criteria
**Rules**
- Be concise
- Avoid generic RCA
```

In [2]:
@mcp.tool()
def create_jira_ticket(
    project_key, title, description, issue_type="Task", priority="Medium", component=None, rca="", fix=""
):
    payload = {
        "fields": {
            "project": {"key": project_key},
            "summary": title,
            "description": {
                "type": "doc",
                "version": 1,
                "content": [
                    {
                        "type": "paragraph",
                        "content": [{"type": "text", "text": f"{description}\n\nRCA:\n{rca}\n\nFix:\n{fix}"}],
                    }
                ],
            },
            "issuetype": {"name": issue_type},
            "priority": {"name": priority},
        }
    }
    if component:
        payload["fields"]["components"] = [{"name": component}]
    return requests.post(f"{JIRA_BASE_URL}/rest/api/3/issue", json=payload, auth=(JIRA_EMAIL, JIRA_API_TOKEN)).json()

# 6. Writing API Endpoints
```
# Skill: Django API Writer (Standards Enforced)
## Objective  
Generate production-ready APIs in Django with strict adherence to organizational standards, ensuring consistency, security, and scalability.
---
## Inputs  
- `endpoint_name`  
- `http_method` (GET, POST, PUT, PATCH, DELETE)  
- `request_schema`  
- `response_schema`  
- `auth_required` (bool)  
- `business_logic_hint` (optional)
---
## Execution Flow  
### 1. Select Base Class
- Always inherit from `APIView` (or org base API class)
### 2. Permissions
- If `auth_required=True` → add `permission_classes`
- Else → explicitly mark as public
### 3. Request Validation
- Use serializers for validation
- Reject invalid inputs with proper error messages
### 4. Business Logic Placement
- Prefer **model/service layer** over API-heavy logic
- Keep views thin and readable
### 5. Response Handling
Return standardized responses:
- `200 OK` → success (GET/PUT/PATCH)
- `201 Created` → resource created
- `400 Bad Request` → validation errors
- `401/403` → auth issues
- `404 Not Found` → resource missing
### 6. Error Handling
- Catch exceptions and return structured error responses
- Avoid exposing internal stack traces
### 7. Code Structure
- Clean, modular, and readable
- Follow consistent naming conventions
### Constraints
Do NOT embed heavy logic inside API views
Always use serializers for validation and response
Enforce permission classes explicitly
Follow organization-specific status code standards
Keep code concise and production-ready
```